# 06_GradCAM_Explainability

Purpose: Generate publication-quality Grad-CAM visualizations for the trained ViT and Swin Transformer models used in the Navarasa FER project.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import timm
from PIL import Image
from torchvision import transforms
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DATASET="/content/drive/MyDrive/split_dataset"
TEST_DIR=os.path.join(DATASET,"test")
VIT_MODEL="/content/drive/MyDrive/Trained Models/vit/vit_best.pth"
SWIN_MODEL="/content/drive/MyDrive/Trained Models/swin/swin_best.pth"
SAVE_DIR="/content/drive/MyDrive/Major_Revision/Explainability"
os.makedirs(SAVE_DIR,exist_ok=True)

In [ ]:
NUM_CLASSES=9
transform=transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

vit_classifier=timm.create_model("vit_base_patch16_224",pretrained=False,num_classes=NUM_CLASSES)
vit_classifier.load_state_dict(torch.load(VIT_MODEL,map_location=device))
vit_classifier.eval().to(device)

swin_classifier=timm.create_model("swin_base_patch4_window7_224",pretrained=False,num_classes=NUM_CLASSES)
swin_classifier.load_state_dict(torch.load(SWIN_MODEL,map_location=device))
swin_classifier.eval().to(device)

In [ ]:
from torchvision.datasets import ImageFolder
test_dataset=ImageFolder(TEST_DIR)
image_path=test_dataset.samples[0][0]
pil_image=Image.open(image_path).convert("RGB")
rgb_img=np.array(pil_image.resize((224,224)))/255.0
input_tensor=transform(pil_image).unsqueeze(0).to(device)

In [ ]:
def reshape_transform_vit(tensor):
    tensor=tensor[:,1:,:]
    h=w=int(tensor.shape[1]**0.5)
    tensor=tensor.reshape(tensor.size(0),h,w,tensor.size(2))
    return tensor.permute(0,3,1,2)

def reshape_transform_swin(x):
    return x.permute(0,3,1,2)

In [ ]:
# ViT Grad-CAM
cam=GradCAM(model=vit_classifier,target_layers=[vit_classifier.blocks[-2].norm1],reshape_transform=reshape_transform_vit)
grayscale_cam=cam(input_tensor=input_tensor)[0]
vit_overlay=show_cam_on_image(rgb_img,grayscale_cam,use_rgb=True)

In [ ]:
# Swin Grad-CAM
cam=GradCAM(model=swin_classifier,target_layers=[swin_classifier.layers[-1].blocks[-1].norm1],reshape_transform=reshape_transform_swin)
grayscale_cam=cam(input_tensor=input_tensor)[0]
swin_overlay=show_cam_on_image(rgb_img,grayscale_cam,use_rgb=True)

In [ ]:
plt.figure(figsize=(18,6))
plt.subplot(131);plt.imshow(rgb_img);plt.title("Original");plt.axis("off")
plt.subplot(132);plt.imshow(vit_overlay);plt.title("ViT Grad-CAM");plt.axis("off")
plt.subplot(133);plt.imshow(swin_overlay);plt.title("Swin Grad-CAM");plt.axis("off")
plt.tight_layout()
out=os.path.join(SAVE_DIR,"Original_ViT_Swin.png")
plt.savefig(out,dpi=300,bbox_inches="tight")
plt.show()
print(out)